## Ads Insights API 수집 가능 지표 확인

In [4]:
"""
Meta Ads Insights API - 특정 광고의 모든 데이터 수집기
=====================================================
아래 변수를 수정해서 사용하세요:
  - ACCESS_TOKEN : Meta API 액세스 토큰
  - AD_ID        : 조회할 광고 ID (act_XXXXXXX 형식이 아닌 순수 ad ID)
  - DATE_PRESET  : 조회 기간 preset (또는 DATE_SINCE/DATE_UNTIL로 커스텀 기간 지정)

실행:
  pip install facebook-business
  python meta_ads_insights_fetcher.py
"""

import json
import logging
from datetime import datetime
from facebook_business.api import FacebookAdsApi
from facebook_business.adobjects.ad import Ad

# =============================================================================
# ✏️  여기를 수정하세요
# =============================================================================
ACCESS_TOKEN = "EAALRoZCi9lYQBQZBWWXu67hjLrkvC17V9K5ZAK4UXvYD7Jtx34PvzhWZArK74Yla5xp0TjW6yd4SikRvu51fOCalZARRo36hgQmFQRqKTA3Uy9jy3LIDDl7C3eGSgKgFbEt4FQwEkqGw2TRgEJGDdvIU2xbEsZAjLQQZCGkCij2Gap9ZBPZCZBU3mzg5el8ZCZBftDCV"
APP_ID       = "793452136469892"         # 없으면 빈 문자열 ""로 두세요
APP_SECRET   = "93c2766fd2a15b7f7997f4586085a4ac"     # 없으면 빈 문자열 ""로 두세요

AD_ID        = "6901235189529"          # 예: "23851234567890"

# 기간 설정 (DATE_PRESET 또는 DATE_SINCE/DATE_UNTIL 둘 중 하나만 사용)
USE_DATE_PRESET = True                    # True면 preset, False면 since/until 사용
DATE_PRESET     = "maximum"              # today / yesterday / last_7d / last_30d / last_90d / this_month / last_month / maximum
DATE_SINCE      = "2025-01-01"            # USE_DATE_PRESET=False일 때 사용
DATE_UNTIL      = "2025-03-25"            # USE_DATE_PRESET=False일 때 사용
# =============================================================================

# 로그 설정
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger(__name__)


# =============================================================================
# 필드 정의
# =============================================================================

# breakdown 없이 조회 가능한 모든 필드
# (breakdown과 함께 요청 불가한 5개 필드는 별도 그룹으로 분리)
FIELDS_WITHOUT_BREAKDOWN = [
    # 기본 식별 정보
    "ad_id",
    "ad_name",
    "adset_id",
    "adset_name",
    "campaign_id",
    "campaign_name",
    "account_id",
    "account_name",

    # 노출·도달
    "impressions",
    "reach",
    "frequency",

    # 클릭
    "clicks",
    "inline_link_clicks",
    "inline_post_engagement",
    "outbound_clicks",
    "unique_clicks",
    "unique_inline_link_clicks",
    "unique_outbound_clicks",

    # 비용
    "spend",
    "cpm",
    "cpc",
    "ctr",
    "cpp",
    "cost_per_inline_link_click",
    "cost_per_unique_click",
    "cost_per_outbound_click",

    # 전환·액션 (action_type breakdown으로 내부 분류됨)
    "actions",
    "action_values",
    "conversions",
    "conversion_values",
    "cost_per_action_type",
    "cost_per_conversion",
    "cost_per_unique_action_type",
    "unique_actions",

    # ROAS
    "purchase_roas",
    "website_purchase_roas",

    # 동영상
    "video_avg_time_watched_actions",
    "video_p25_watched_actions",
    "video_p50_watched_actions",
    "video_p75_watched_actions",
    "video_p95_watched_actions",
    "video_p100_watched_actions",
    "video_thruplay_watched_actions",
    "video_30_sec_watched_actions",
    "video_continuous_2_sec_watched_actions",
    "video_play_actions",
    "video_play_curve_actions",

    # 광고 상기
    "estimated_ad_recallers",
    "estimated_ad_recall_rate",

    # 소셜
    "social_spend",

    # 기타
    "objective",
    "buying_type",
    "quality_ranking",
    "engagement_rate_ranking",
    "conversion_rate_ranking",
]

# breakdown 없이 단독으로만 요청 가능한 필드 (breakdown과 함께 요청하면 API 오류)
FIELDS_NO_BREAKDOWN_ONLY = [
    "app_store_clicks",
    "newsfeed_avg_position",
    "newsfeed_clicks",
    "newsfeed_impressions",
    # relevance_score는 완전 deprecated되어 제외
]

# age+gender breakdown과 함께 요청 가능한 필드
# (reach/unique_* 는 포함하되 13개월 제한 주의 - 30일 기간이면 문제없음)
FIELDS_WITH_BREAKDOWN = [
    "impressions",
    "reach",
    "clicks",
    "inline_link_clicks",
    "spend",
    "cpm",
    "cpc",
    "ctr",
    "cpp",
    "actions",
    "action_values",
    "conversions",
    "conversion_values",
    "cost_per_action_type",
    "cost_per_conversion",
    "purchase_roas",
    "website_purchase_roas",
    "unique_clicks",
    "unique_actions",
    "unique_ctr",
    "estimated_ad_recallers",
    "estimated_ad_recall_rate",
    "video_avg_time_watched_actions",
    "video_p25_watched_actions",
    "video_p50_watched_actions",
    "video_p75_watched_actions",
    "video_p95_watched_actions",
    "video_p100_watched_actions",
    "video_thruplay_watched_actions",
    "video_30_sec_watched_actions",
]


# =============================================================================
# 유틸
# =============================================================================

def build_params(extra: dict = None) -> dict:
    """공통 파라미터 빌드"""
    params = {}
    if USE_DATE_PRESET:
        params["date_preset"] = DATE_PRESET
    else:
        params["time_range"] = {"since": DATE_SINCE, "until": DATE_UNTIL}
    if extra:
        params.update(extra)
    return params


def pretty(data) -> str:
    """JSON 예쁘게 출력"""
    return json.dumps(data, indent=2, ensure_ascii=False, default=str)


def log_section(title: str):
    log.info("=" * 70)
    log.info(f"  {title}")
    log.info("=" * 70)


def fetch_insights(ad: Ad, fields: list, params: dict, label: str) -> list:
    """insights 조회 후 로그 출력, 결과 반환"""
    log_section(label)
    log.info(f"요청 fields  : {fields}")
    log.info(f"요청 params  : {pretty(params)}")
    try:
        insights = ad.get_insights(fields=fields, params=params)
        results = [dict(row) for row in insights]
        if results:
            log.info(f"응답 ({len(results)}건):\n{pretty(results)}")
        else:
            log.warning("응답 데이터 없음 (해당 기간 데이터 없거나 광고 미게재)")
        return results
    except Exception as e:
        log.error(f"API 오류: {e}")
        return []


# =============================================================================
# 메인
# =============================================================================

def main():
    log.info(f"Meta Ads Insights 수집 시작 - {datetime.now().isoformat()}")
    log.info(f"AD_ID        : {AD_ID}")
    log.info(f"기간         : {DATE_PRESET if USE_DATE_PRESET else f'{DATE_SINCE} ~ {DATE_UNTIL}'}")

    # API 초기화
    FacebookAdsApi.init(
        app_id=APP_ID or None,
        app_secret=APP_SECRET or None,
        access_token=ACCESS_TOKEN,
    )
    ad = Ad(AD_ID)

    # ------------------------------------------------------------------
    # 0. API 버전 확인
    # ------------------------------------------------------------------
    api = FacebookAdsApi.get_default_api()
    log.info(f"사용 중인 Meta API 버전: {api.API_VERSION}")

    # ------------------------------------------------------------------
    # 1. breakdown 없이 — 전체 지표
    # ------------------------------------------------------------------
    fetch_insights(
        ad=ad,
        fields=FIELDS_WITHOUT_BREAKDOWN,
        params=build_params(),
        label="[1/3] breakdown 없음 — 전체 지표",
    )

    # ------------------------------------------------------------------
    # 2. breakdown 없이만 가능한 필드 (newsfeed_*, app_store_clicks)
    # ------------------------------------------------------------------
    fetch_insights(
        ad=ad,
        fields=FIELDS_NO_BREAKDOWN_ONLY,
        params=build_params(),
        label="[2/3] breakdown 없이만 조회 가능한 필드 (newsfeed_* / app_store_clicks)",
    )

    # ------------------------------------------------------------------
    # 3. age + gender breakdown
    # ------------------------------------------------------------------
    fetch_insights(
        ad=ad,
        fields=FIELDS_WITH_BREAKDOWN,
        params=build_params({"breakdowns": ["age", "gender"]}),
        label="[3/3] age + gender breakdown",
    )

    log.info("=" * 70)
    log.info(f"수집 완료 - {datetime.now().isoformat()}")
    log.info("=" * 70)

    # ------------------------------------------------------------------
    # 참고: breakdown 값의 합산 ≠ 전체값 주의사항 출력
    # ------------------------------------------------------------------
    log.warning(
        "⚠️  reach / unique_* 계열은 breakdown 합산값 ≠ 전체값입니다. "
        "고유 사용자 중복 제거 방식 차이로 인해 breakdown별 reach 합산이 "
        "전체 reach보다 크게 나올 수 있습니다."
    )


if __name__ == "__main__":
    main()

2026-03-26 18:34:58 [INFO] Meta Ads Insights 수집 시작 - 2026-03-26T18:34:58.306269
2026-03-26 18:34:58 [INFO] AD_ID        : 6901235189529
2026-03-26 18:34:58 [INFO] 기간         : maximum
2026-03-26 18:34:58 [INFO] 사용 중인 Meta API 버전: v25.0
2026-03-26 18:34:58 [INFO] ======================================================================
2026-03-26 18:34:58 [INFO]   [1/3] breakdown 없음 — 전체 지표
2026-03-26 18:34:58 [INFO] ======================================================================
2026-03-26 18:34:58 [INFO] 요청 fields  : ['ad_id', 'ad_name', 'adset_id', 'adset_name', 'campaign_id', 'campaign_name', 'account_id', 'account_name', 'impressions', 'reach', 'frequency', 'clicks', 'inline_link_clicks', 'inline_post_engagement', 'outbound_clicks', 'unique_clicks', 'unique_inline_link_clicks', 'unique_outbound_clicks', 'spend', 'cpm', 'cpc', 'ctr', 'cpp', 'cost_per_inline_link_click', 'cost_per_unique_click', 'cost_per_outbound_click', 'actions', 'action_values', 'conversions', 'conversion_val

## Instagram Insights API 수집 가능 지표 확인

In [5]:
"""
Instagram Insights API - 계정 & 게시물 전체 지표 수집기
=======================================================
아래 변수를 수정해서 사용하세요:
  - ACCESS_TOKEN   : Meta API 액세스 토큰
  - IG_USER_ID     : Instagram 비즈니스/크리에이터 계정 ID
  - MEDIA_IDS      : 조회할 게시물 ID 목록 (여러 개 가능)

실행:
  pip install facebook-business
  python instagram_insights_fetcher.py
"""

import json
import logging
from datetime import datetime, timedelta
from facebook_business.api import FacebookAdsApi
from facebook_business.adobjects.iguser import IGUser
from facebook_business.adobjects.igmedia import IGMedia

# =============================================================================
# ✏️  여기를 수정하세요
# =============================================================================
ACCESS_TOKEN = "EAALRoZCi9lYQBQZBWWXu67hjLrkvC17V9K5ZAK4UXvYD7Jtx34PvzhWZArK74Yla5xp0TjW6yd4SikRvu51fOCalZARRo36hgQmFQRqKTA3Uy9jy3LIDDl7C3eGSgKgFbEt4FQwEkqGw2TRgEJGDdvIU2xbEsZAjLQQZCGkCij2Gap9ZBPZCZBU3mzg5el8ZCZBftDCV"
APP_ID       = "793452136469892"   # 없으면 빈 문자열
APP_SECRET   = "93c2766fd2a15b7f7997f4586085a4ac"   # 없으면 빈 문자열

IG_USER_ID   = "17841403964728081"      # 예: "17841405822304914"

MEDIA_IDS    = [
    "17978120294869424",                      # 피드 게시물 ID
    # "YOUR_MEDIA_ID_2",                    # 릴스 ID (필요시 추가)
    # "YOUR_MEDIA_ID_3",                    # 스토리 ID (필요시 추가)
]

# 계정 인사이트 조회 기간 (day period 기준)
SINCE_DAYS_AGO = 30   # 오늘 기준 며칠 전부터
# =============================================================================


# 로그 설정
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger(__name__)


# =============================================================================
# 유틸
# =============================================================================

def pretty(data) -> str:
    return json.dumps(data, indent=2, ensure_ascii=False, default=str)

def log_section(title: str):
    log.info("=" * 70)
    log.info(f"  {title}")
    log.info("=" * 70)

def since_until():
    """최근 N일 unix timestamp 반환"""
    now   = datetime.utcnow()
    since = int((now - timedelta(days=SINCE_DAYS_AGO)).timestamp())
    until = int(now.timestamp())
    return since, until


# =============================================================================
# 계정(User) 인사이트
# =============================================================================

def fetch_user_interaction_metrics(ig_user: IGUser):
    """
    계정 레벨 인터랙션 지표 — period=day, metric_type=total_value
    breakdown 가능한 지표: reach(media_product_type, follow_type), views(follower_type, media_product_type)
    """
    since, until = since_until()

    # --- 1) breakdown 없는 인터랙션 지표 전체 ---
    log_section("[계정 1/5] 인터랙션 지표 — breakdown 없음")
    metrics_no_breakdown = [
        "accounts_engaged",
        "reach",
        "views",
        "impressions",          # v22.0+ deprecated, 하위 버전 호환용
        "follows_and_unfollows",
        "likes",
        "comments",
        "shares",
        "saves",
        "replies",
        "reposts",
        "profile_links_taps",
        "total_interactions",
    ]
    params = {
        "metric": ",".join(metrics_no_breakdown),
        "period": "day",
        "metric_type": "total_value",
        "since": since,
        "until": until,
    }
    log.info(f"요청 params: {pretty(params)}")
    try:
        result = ig_user.get_insights(params=params)
        data = [dict(r) for r in result]
        log.info(f"응답 ({len(data)}건):\n{pretty(data)}")
    except Exception as e:
        log.error(f"API 오류: {e}")

    # --- 2) reach — media_product_type breakdown ---
    log_section("[계정 2/5] reach — breakdown: media_product_type")
    params = {
        "metric": "reach",
        "period": "day",
        "metric_type": "total_value",
        "breakdown": "media_product_type",
        "since": since,
        "until": until,
    }
    log.info(f"요청 params: {pretty(params)}")
    try:
        result = ig_user.get_insights(params=params)
        data = [dict(r) for r in result]
        log.info(f"응답 ({len(data)}건):\n{pretty(data)}")
    except Exception as e:
        log.error(f"API 오류: {e}")

    # --- 3) reach — follow_type breakdown ---
    log_section("[계정 3/5] reach — breakdown: follow_type")
    params = {
        "metric": "reach",
        "period": "day",
        "metric_type": "total_value",
        "breakdown": "follow_type",
        "since": since,
        "until": until,
    }
    log.info(f"요청 params: {pretty(params)}")
    try:
        result = ig_user.get_insights(params=params)
        data = [dict(r) for r in result]
        log.info(f"응답 ({len(data)}건):\n{pretty(data)}")
    except Exception as e:
        log.error(f"API 오류: {e}")

    # --- 4) views — follower_type + media_product_type breakdown ---
    log_section("[계정 4/5] views — breakdown: follower_type, media_product_type")
    for bd in ["follower_type", "media_product_type"]:
        params = {
            "metric": "views",
            "period": "day",
            "metric_type": "total_value",
            "breakdown": bd,
            "since": since,
            "until": until,
        }
        log.info(f"breakdown={bd} 요청 params: {pretty(params)}")
        try:
            result = ig_user.get_insights(params=params)
            data = [dict(r) for r in result]
            log.info(f"응답 ({len(data)}건):\n{pretty(data)}")
        except Exception as e:
            log.error(f"API 오류: {e}")


def fetch_user_demographic_metrics(ig_user: IGUser):
    """
    계정 레벨 인구통계 지표 — period=lifetime, timeframe 필수
    breakdown: age / gender / city / country
    """
    timeframe = "last_30_days"

    # --- 5) 인구통계 지표 — 각 breakdown별 ---
    log_section("[계정 5/5] 인구통계 지표 — engaged_audience_demographics & follower_demographics")

    for metric in ["engaged_audience_demographics", "follower_demographics"]:
        for breakdown in ["age", "gender", "city", "country"]:
            params = {
                "metric": metric,
                "period": "lifetime",
                "timeframe": timeframe,
                "metric_type": "total_value",
                "breakdown": breakdown,
            }
            log.info(f"[{metric} / breakdown={breakdown}] 요청 params: {pretty(params)}")
            try:
                result = ig_user.get_insights(params=params)
                data = [dict(r) for r in result]
                log.info(f"응답 ({len(data)}건):\n{pretty(data)}")
            except Exception as e:
                log.error(f"API 오류: {e}")


# =============================================================================
# 미디어(Media) 인사이트
# =============================================================================

# 미디어 타입별 지원 메트릭 정의
# (어떤 타입인지 모를 경우 전부 시도하고 오류 로그로 확인)
MEDIA_METRICS_COMMON = [
    "reach",
    "likes",
    "comments",
    "shares",
    "saved",
    "total_interactions",
    "views",
    "follows",
    "profile_visits",
    "profile_activity",   # breakdown: action_type 지원
]

MEDIA_METRICS_FEED_REELS = [
    # FEED / REELS 전용
]

MEDIA_METRICS_STORY = [
    "replies",
    "navigation",         # breakdown: story_navigation_action_type 지원
]

MEDIA_METRICS_REELS = [
    "ig_reels_avg_watch_time",
    "ig_reels_video_view_total_time",
    # deprecated (요청은 해보되 오류 예상)
    "plays",
    "clips_replays_count",
    "ig_reels_aggregated_all_plays_count",
]

MEDIA_METRICS_DEPRECATED = [
    # 2025.4.21 전면 폐기 — 오류 예상, 확인용으로만 포함
    "impressions",
    "video_views",
]


def fetch_media_insights(media_id: str):
    media = IGMedia(media_id)

    # 미디어 기본 정보 조회 (타입 확인용)
    log_section(f"[미디어 {media_id}] 기본 정보")
    try:
        info = media.api_get(fields=["id", "media_type", "media_product_type", "timestamp", "caption"])
        log.info(f"미디어 정보:\n{pretty(dict(info))}")
    except Exception as e:
        log.error(f"미디어 정보 조회 오류: {e}")

    # --- 공통 지표 — breakdown 없음 ---
    log_section(f"[미디어 {media_id}] 공통 지표 — breakdown 없음")
    _fetch_media_metric(media, MEDIA_METRICS_COMMON + MEDIA_METRICS_DEPRECATED, label="공통+deprecated")

    # --- 스토리 전용 지표 ---
    log_section(f"[미디어 {media_id}] 스토리 전용 지표")
    _fetch_media_metric(media, MEDIA_METRICS_STORY, label="스토리")

    # --- 릴스 전용 지표 ---
    log_section(f"[미디어 {media_id}] 릴스 전용 지표")
    _fetch_media_metric(media, MEDIA_METRICS_REELS, label="릴스")

    # --- profile_activity — breakdown: action_type ---
    log_section(f"[미디어 {media_id}] profile_activity — breakdown: action_type")
    params = {
        "metric": "profile_activity",
        "breakdown": "action_type",
    }
    log.info(f"요청 params: {pretty(params)}")
    try:
        result = media.get_insights(params=params)
        data = [dict(r) for r in result]
        log.info(f"응답 ({len(data)}건):\n{pretty(data)}")
    except Exception as e:
        log.error(f"API 오류: {e}")

    # --- navigation — breakdown: story_navigation_action_type ---
    log_section(f"[미디어 {media_id}] navigation — breakdown: story_navigation_action_type")
    params = {
        "metric": "navigation",
        "breakdown": "story_navigation_action_type",
    }
    log.info(f"요청 params: {pretty(params)}")
    try:
        result = media.get_insights(params=params)
        data = [dict(r) for r in result]
        log.info(f"응답 ({len(data)}건):\n{pretty(data)}")
    except Exception as e:
        log.error(f"API 오류: {e}")


def _fetch_media_metric(media: IGMedia, metrics: list, label: str):
    """미디어 지표를 한 번에 요청. 오류 시 개별 재시도"""
    params = {"metric": ",".join(metrics)}
    log.info(f"[{label}] 요청 metrics: {metrics}")
    try:
        result = media.get_insights(params=params)
        data = [dict(r) for r in result]
        log.info(f"응답 ({len(data)}건):\n{pretty(data)}")
    except Exception as e:
        log.warning(f"일괄 요청 실패 ({e}) → 개별 재시도")
        for m in metrics:
            try:
                result = media.get_insights(params={"metric": m})
                data = [dict(r) for r in result]
                log.info(f"[{m}] 응답 ({len(data)}건):\n{pretty(data)}")
            except Exception as e2:
                log.error(f"[{m}] 오류: {e2}")


# =============================================================================
# 메인
# =============================================================================

def main():
    log.info(f"Instagram Insights 수집 시작 - {datetime.now().isoformat()}")
    log.info(f"IG_USER_ID : {IG_USER_ID}")
    log.info(f"MEDIA_IDS  : {MEDIA_IDS}")
    log.info(f"기간       : 최근 {SINCE_DAYS_AGO}일")

    # API 초기화
    FacebookAdsApi.init(
        app_id=APP_ID or None,
        app_secret=APP_SECRET or None,
        access_token=ACCESS_TOKEN,
    )

    # API 버전 확인
    api = FacebookAdsApi.get_default_api()
    log.info(f"Meta API 버전: {api.API_VERSION}")

    ig_user = IGUser(IG_USER_ID)

    # ── 계정 인사이트 ──────────────────────────────────────────────────────
    log_section("■ 계정(User) 인사이트")
    fetch_user_interaction_metrics(ig_user)
    fetch_user_demographic_metrics(ig_user)

    # ── 미디어 인사이트 ────────────────────────────────────────────────────
    for media_id in MEDIA_IDS:
        log_section(f"■ 미디어 인사이트: {media_id}")
        fetch_media_insights(media_id)

    log.info("=" * 70)
    log.info(f"수집 완료 - {datetime.now().isoformat()}")
    log.info("=" * 70)

    log.warning(
        "⚠️  스토리 지표는 발행 후 24시간 이내에만 조회 가능합니다. "
        "만료된 스토리 ID는 오류가 반환됩니다."
    )
    log.warning(
        "⚠️  인구통계(engaged_audience_demographics / follower_demographics)는 "
        "팔로워 100명 미만 계정에서는 반환되지 않습니다."
    )
    log.warning(
        "⚠️  impressions / plays / clips_replays_count / ig_reels_aggregated_all_plays_count는 "
        "2025년 4월 21일부로 전면 폐기된 지표입니다. 오류가 반환될 수 있습니다."
    )


if __name__ == "__main__":
    main()

2026-03-27 08:28:00 [INFO] Instagram Insights 수집 시작 - 2026-03-27T08:28:00.944732
2026-03-27 08:28:00 [INFO] IG_USER_ID : 17841403964728081
2026-03-27 08:28:00 [INFO] MEDIA_IDS  : ['17978120294869424']
2026-03-27 08:28:00 [INFO] 기간       : 최근 30일
2026-03-27 08:28:00 [INFO] Meta API 버전: v25.0
2026-03-27 08:28:00 [INFO] ======================================================================
2026-03-27 08:28:00 [INFO]   ■ 계정(User) 인사이트
2026-03-27 08:28:00 [INFO] ======================================================================
2026-03-27 08:28:00 [INFO] ======================================================================
2026-03-27 08:28:00 [INFO]   [계정 1/5] 인터랙션 지표 — breakdown 없음
2026-03-27 08:28:00 [INFO] ======================================================================
2026-03-27 08:28:00 [INFO] 요청 params: {
  "metric": "accounts_engaged,reach,views,impressions,follows_and_unfollows,likes,comments,shares,saves,replies,reposts,profile_links_taps,total_interactions",
  "period"